# 04 Temperature-sorted intensity

Tercera fase del flujo: para una muestra puntual, muestra la intensidad normalizada (`NormSignal`) en funcion de la temperatura de las ROI seleccionadas (`ROI_status == 1`), igual que en `03_analysis.ipynb` pero a nivel de una sola muestra, y ademas genera, por cada ROI seleccionada, un `.tif` multi-frame con los frames reordenados de menor a mayor temperatura (heating y cooling mezclados) en vez del orden de adquisicion original. Sirve para ver, como pelicula en ImageJ, como cambia la intensidad de senal a medida que sube la temperatura.

No recalcula normalizacion, `phase`, `trend` ni `ROI_status`: parte de los `*_preprocessed_long.csv` ya curados en `01_preprocessing.ipynb`.

In [1]:
%matplotlib qt
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(str(Path("../scripts").resolve()))
import analisis_ttl as ttl
import roi_image_crops as crops

base_dir = Path("/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data")

## Parametros

`folder` es el nombre EXACTO de una carpeta de muestra (source_folder), ej. `mut45_image13`. Todo lo de abajo (curva por temperatura, tif ordenado) usa esta misma muestra.

In [2]:
folder = "mut27_image11"

# phase y trend no se filtran aca: ya vienen fijados en los *_preprocessed_long.csv
# (fase 01_preprocessing.ipynb), y ROI_status ya refleja la curaduria manual
# sobre esa clasificacion. Filtrar de nuevo seria redundante.

# Filtro opcional solo para la curva de NormSignal vs temperatura (no afecta
# que ROI se usan para el tif ordenado, que sigue siendo ROI_status==1).
temp_range = None  # ej. (25, 40); None conserva todo el rango
value_col = "NormSignal"

# Tif ordenado por temperatura
box_size = 20
output_subfolder = "ROIs_by_temp"
save_format = "tif"

# Collage (montage): en vez de mostrar los 77 frames originales (varios
# practicamente identicos donde la curva es plana), se elige 1 frame cada
# temp_bin_width °C, y se arma como una tira larga (una fila), igual de
# ancha que el rango de temperatura de la muestra -> mas representativa de
# la curva NormSignal vs temperatura de arriba.
temp_bin_width = 0.5

## Cargar `_long` compilado

Igual que en `02_processing.ipynb`/`03_analysis.ipynb`: concatena todos los `*_preprocessed_long.csv` bajo `base_dir`, sin depender de ningun Excel.

In [3]:
imports = ttl.load_all_preprocessed_long(base_dir)

load_status = imports["load_status"]
preprocessed_all = imports["preprocessed_all"]

print(f"preprocessed_all: {preprocessed_all.shape}")
if preprocessed_all.empty:
    print("No se encontraron archivos *_preprocessed_long.csv. Ejecuta primero 01_preprocessing.ipynb para cada muestra.")

duplicated_preprocessed = load_status[load_status["n_files"] > 1]
if not duplicated_preprocessed.empty:
    print("ADVERTENCIA: carpetas con mas de un *_preprocessed_long.csv (revisar y dejar solo el correcto):")
    print(duplicated_preprocessed[["folder", "preprocessed_files"]].to_string(index=False))

if folder not in set(preprocessed_all["source_folder"].dropna().unique()):
    raise ValueError(f"folder={folder!r} no esta en preprocessed_all. Carpetas disponibles: {sorted(preprocessed_all['source_folder'].dropna().unique())}")

sample_df = preprocessed_all[preprocessed_all["source_folder"] == folder].copy()
selected_rois = sorted(
    sample_df.loc[sample_df["ROI_status"] == 1, "ROI"].unique(),
    key=lambda r: int(str(r).replace("ROI", "")),
)
genotype = sample_df["genotype"].dropna().iloc[0]
sample = sample_df["sample"].dropna().iloc[0]

print(f"folder: {folder} | genotype: {genotype} | sample: {sample}")
print(f"ROI seleccionadas (ROI_status==1): {len(selected_rois)}")
print(selected_rois)

preprocessed_all: (236630, 43)
folder: mut27_image11 | genotype: m27 | sample: sample_11
ROI seleccionadas (ROI_status==1): 62
['ROI8', 'ROI10', 'ROI14', 'ROI16', 'ROI18', 'ROI23', 'ROI26', 'ROI35', 'ROI52', 'ROI53', 'ROI60', 'ROI61', 'ROI65', 'ROI87', 'ROI88', 'ROI90', 'ROI100', 'ROI104', 'ROI106', 'ROI109', 'ROI114', 'ROI125', 'ROI129', 'ROI132', 'ROI157', 'ROI170', 'ROI175', 'ROI177', 'ROI192', 'ROI201', 'ROI210', 'ROI212', 'ROI213', 'ROI217', 'ROI234', 'ROI236', 'ROI239', 'ROI242', 'ROI244', 'ROI255', 'ROI257', 'ROI259', 'ROI264', 'ROI265', 'ROI270', 'ROI276', 'ROI291', 'ROI297', 'ROI303', 'ROI306', 'ROI308', 'ROI309', 'ROI313', 'ROI322', 'ROI323', 'ROI324', 'ROI333', 'ROI335', 'ROI339', 'ROI353', 'ROI354', 'ROI356']


## Intensidad normalizada vs temperatura (ROI seleccionadas)

Reutiliza `ttl.graph_selected_rois_by_folder`, filtrando antes por `phase_filter`/`trend_filter`/`temp_range` (la seleccion de ROI sigue siendo `ROI_status == 1`, fijada en preprocesamiento).

In [4]:
curve_df = sample_df.copy()

if temp_range is not None:
    temp_min, temp_max = temp_range
    curve_df = curve_df[curve_df["temp_mean"].between(temp_min, temp_max, inclusive="both")]

plotted = ttl.graph_selected_rois_by_folder(
    curve_df,
    folder=folder,
    value_col=value_col,
    show_mean_se=True,
)

mut27_image11: 62 ROIs graficadas, 7812 filas


/Users/gfernandezv/Documents/envs/Images_TTL_temp/scripts/analisis_ttl.py:635: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


## Frame -> temperatura de la muestra

La temperatura es una propiedad del frame, no de la ROI: esta tabla (un `frame` por fila) es la misma para todas las ROI de `folder` y es la que se usa para reordenar el tif por temperatura.

In [5]:
frame_temp_df = (
    sample_df[["frame", "temp_mean", "phase", "trend"]]
    .drop_duplicates(subset="frame")
    .sort_values("frame")
    .reset_index(drop=True)
)

print(f"frame_temp_df: {frame_temp_df.shape}")
display(frame_temp_df.head())

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(frame_temp_df["frame"], frame_temp_df["temp_mean"], marker="o", markersize=3, linewidth=1)
ax.set_xlabel("frame (orden de adquisicion)")
ax.set_ylabel("temp_mean (°C)")
ax.set_title(f"{folder}: temperatura por frame")
fig.tight_layout()
plt.show()

frame_temp_df: (126, 4)


,frame,temp_mean,phase,trend
0,19,21.560243,cooling,increase
1,20,21.530334,cooling,increase
2,21,21.279992,cooling,increase
3,22,21.179078,cooling,increase
4,23,21.152815,cooling,increase


## Tif ordenado por temperatura (todas las ROI seleccionadas)

Para cada ROI con `ROI_status == 1` en `folder`, genera un recorte `box_size x box_size` con TODOS los frames del stack original, pero reordenados de menor a mayor `temp_mean` (heating y cooling mezclados, sin separar por fase). Se guarda en `<sample_dir>/<output_subfolder>/`, con sufijo `_sorted_by_temp` para no pisar los recortes en orden original de `crop_included_rois_from_notebook` (carpeta `ROIs/`).

In [6]:
sample_dir = base_dir / folder
output_dir = sample_dir / output_subfolder

results = crops.crop_selected_rois_sorted_by_temperature(
    sample_dir=sample_dir,
    selected_roi_names=selected_rois,
    frame_temp_df=frame_temp_df,
    output_dir=output_dir,
    genotype=genotype,
    sample=sample,
    box_size=box_size,
    save_format=save_format,
)

results

Recortes ordenados por temperatura generados: 62 / 62 en /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp


{'ROI8': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R008_sorted_by_temp.tif'),
 'ROI10': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R010_sorted_by_temp.tif'),
 'ROI14': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R014_sorted_by_temp.tif'),
 'ROI16': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R016_sorted_by_temp.tif'),
 'ROI18': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R018_sorted_by_temp.tif'),
 'ROI23': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/m27_i11_R023_sorted_by_temp.tif'),
 'ROI26': PosixPath('/Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp/

## Collage de frames ordenados por temperatura (sin visor de TIFF)

Python no trae un visor de TIFF interactivo equivalente al de ImageJ/Fiji. Como los recortes son chicos (`box_size x box_size`), en vez de instalar un visor (ej. napari) armamos un collage: cada frame del `.tif` ordenado por temperatura queda como una celda de una grilla, en una sola imagen PNG. Se guarda junto al `.tif` en `output_dir`, con sufijo `_montage.png`.

In [7]:
montage_paths = {}

temp_sorted = frame_temp_df.sort_values("temp_mean").reset_index(drop=True)

for roi_name, tif_path in results.items():
    if tif_path is None:
        montage_paths[roi_name] = None
        continue

    import tifffile
    sorted_crop = tifffile.imread(str(tif_path))

    picked_idx = crops.select_indices_by_temp_bin(temp_sorted["temp_mean"].to_numpy(), bin_width=temp_bin_width)
    binned_crop = sorted_crop[picked_idx]

    png_path = tif_path.with_name(tif_path.stem + "_montage.png")
    crops.save_montage_png(binned_crop, png_path, ncols=len(picked_idx))
    montage_paths[roi_name] = png_path

n_ok = sum(1 for v in montage_paths.values() if v is not None)
print(f"Collages generados: {n_ok} / {len(results)} en {output_dir} (1 frame cada {temp_bin_width} °C)")

# Vista previa de un ROI (el primero con collage generado)
example_roi = next((r for r, p in montage_paths.items() if p is not None), None)
if example_roi is not None:
    import imageio.v3 as iio
    preview = iio.imread(montage_paths[example_roi])
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.imshow(preview, cmap="gray")
    ax.set_title(f"{folder} | {example_roi}: 1 frame cada {temp_bin_width} °C ({frame_temp_df['temp_mean'].min():.1f}-{frame_temp_df['temp_mean'].max():.1f} °C)")
    ax.axis("off")
    fig.tight_layout()
    plt.show()

Collages generados: 62 / 62 en /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/mut27_image11/ROIs_by_temp (1 frame cada 0.5 °C)


/Users/gfernandezv/miniconda3/envs/ttl_imagej/lib/python3.11/site-packages/ipykernel/eventloops.py:145: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  el.exec() if hasattr(el, "exec") else el.exec_()
